# BioRob Phase 5 — LOSO Exporter FINAL Sliding-Only Overwrite

This notebook rebuilds Phase 5 exports after Phase 3 was changed to `ENABLE_ONSET_ANCHOR_WINDOWS = False`. It backs up/removes old `exports_v1_balanced_foldK` and `exports_v1_ssl_foldK` folders before regenerating sliding-only exports.

In [11]:
# =====================================================================
# Phase 5 — LOSO Exporter FINAL (BioRob)
# ---------------------------------------------------------------------
# Input:
#   /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/manifest_v1.csv
#   /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/splits_v1.csv
#   Phase 4 caches next to Phase 2B labelonly CSVs:
#      *.csv.preproc.npz
#
# Output:
#   _dataset_icml_v1/exports_v1_balanced_foldK/
#   _dataset_icml_v1/exports_v1_ssl_foldK/
#
# Important:
#   - This code uses only Phase 3 split rows and Phase 4 caches.
#   - The 104 missing Phase 1C files are already excluded from Phase 3.
#   - This code does not create or modify input CSV columns.
#   - It validates required split/cache fields before export.
# =====================================================================

from __future__ import annotations

import json
import math
import re
import shutil
import warnings
from datetime import datetime
from pathlib import Path
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [12]:
# =====================================================================
# CELL 1 — Configuration
# =====================================================================

ROOT_DIR = Path("/home/tsultan1/BioRob/Human Subject Data")
DATASET_DIR = ROOT_DIR / "_dataset_icml_v1"

MANIFEST_CSV = DATASET_DIR / "manifest_v1.csv"
SPLITS_CSV   = DATASET_DIR / "splits_v1.csv"

# Optional Phase 4.5 QC file. If it does not exist, no QC filter is applied.
QC_CSV = DATASET_DIR / "qc_summary_v1.csv"

AUDIT_DIR = ROOT_DIR / "_audit_phase5_exporter"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Export names expected by Phase 6.
OUT_NAME_BALANCED = "exports_v1_balanced"
OUT_NAME_SSL      = "exports_v1_ssl"

# Run controls
RUN_INPUT_AUDIT = True

# Recommended first run:
#   RUN_ONE_FOLD_FIRST=True, RUN_ALL_FOLDS=False
# Final run:
#   RUN_ONE_FOLD_FIRST=False, RUN_ALL_FOLDS=True
RUN_ONE_FOLD_FIRST = False
RUN_ALL_FOLDS = True
ONE_FOLD_ID = 1

EXPORT_BALANCED = True
EXPORT_SSL = True

# If False, completed fold export folders are skipped.
# If True, old export folders are deleted and rebuilt.
OVERWRITE_EXPORTS = True

# Safer final rerun behavior after changing to sliding-only windows.
# Old export folders are moved to a timestamped backup before new exports are created.
# This prevents accidentally mixing old sliding+onset_anchor shards with new sliding-only shards.
BACKUP_OLD_EXPORTS_BEFORE_OVERWRITE = True
CLEAN_PHASE5_EXPORTS_BEFORE_RUN = True

# Window/export settings
WINDOW_TYPES = ["sliding"]
INCLUDE_EEG, INCLUDE_EMG, INCLUDE_ET = True, True, True

SHARD_SIZE = 5000
FIXED_WINDOW_LEN = 500
MIN_COVERAGE = 0.40

# Balanced supervised export only.
MAX_TASK0_RATIO = 0.20
MIN_TASK_SAMPLES = 50
ACTION_MAX_FACTOR = 3

# Optional caps for quick debugging. Keep None for final export.
LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

# Expected Phase 4 NPZ keys. These are arrays saved by Phase 4.
REQUIRED_CACHE_KEYS = [
    "fs", "t",
    "EEG", "EEG_mask", "EEG_ch",
    "EMG_env", "EMG_mask", "EMG_ch",
    "ET", "ET_mask", "ET_ch",
    "labels", "label_ch",
    "subject_id", "task", "trial",
]

EEG_KEY      = "EEG"
EEG_MASK_KEY = "EEG_mask"
EEG_CH_KEY   = "EEG_ch"

EMG_KEY      = "EMG_env"
EMG_MASK_KEY = "EMG_mask"
EMG_CH_KEY   = "EMG_ch"

ET_KEY       = "ET"
ET_MASK_KEY  = "ET_mask"
ET_CH_KEY    = "ET_ch"

LABEL_KEY    = "labels"
LABEL_CH_KEY = "label_ch"

NPZ_SUFFIX = ".preproc.npz"

print("ROOT_DIR    :", ROOT_DIR)
print("DATASET_DIR :", DATASET_DIR)
print("MANIFEST_CSV:", MANIFEST_CSV)
print("SPLITS_CSV  :", SPLITS_CSV)
print("QC_CSV      :", QC_CSV)
print("AUDIT_DIR   :", AUDIT_DIR)
print()
print("RUN_INPUT_AUDIT   =", RUN_INPUT_AUDIT)
print("RUN_ONE_FOLD_FIRST=", RUN_ONE_FOLD_FIRST)
print("RUN_ALL_FOLDS     =", RUN_ALL_FOLDS)
print("ONE_FOLD_ID       =", ONE_FOLD_ID)
print("EXPORT_BALANCED   =", EXPORT_BALANCED)
print("EXPORT_SSL        =", EXPORT_SSL)
print("OVERWRITE_EXPORTS =", OVERWRITE_EXPORTS)
print("BACKUP_OLD_EXPORTS_BEFORE_OVERWRITE =", BACKUP_OLD_EXPORTS_BEFORE_OVERWRITE)
print("CLEAN_PHASE5_EXPORTS_BEFORE_RUN =", CLEAN_PHASE5_EXPORTS_BEFORE_RUN)
print("WINDOW_TYPES =", WINDOW_TYPES)

ROOT_DIR    : /home/tsultan1/BioRob/Human Subject Data
DATASET_DIR : /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1
MANIFEST_CSV: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/manifest_v1.csv
SPLITS_CSV  : /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/splits_v1.csv
QC_CSV      : /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/qc_summary_v1.csv
AUDIT_DIR   : /home/tsultan1/BioRob/Human Subject Data/_audit_phase5_exporter

RUN_INPUT_AUDIT   = True
RUN_ONE_FOLD_FIRST= False
RUN_ALL_FOLDS     = True
ONE_FOLD_ID       = 1
EXPORT_BALANCED   = True
EXPORT_SSL        = True
OVERWRITE_EXPORTS = True
BACKUP_OLD_EXPORTS_BEFORE_OVERWRITE = True
CLEAN_PHASE5_EXPORTS_BEFORE_RUN = True
WINDOW_TYPES = ['sliding']


In [13]:
# =====================================================================
# CELL 2 — Load manifest and splits
# =====================================================================

if not MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_CSV}")

if not SPLITS_CSV.exists():
    raise FileNotFoundError(f"Splits not found: {SPLITS_CSV}")

manifest = pd.read_csv(MANIFEST_CSV)
splits = pd.read_csv(SPLITS_CSV)

print("\n" + "=" * 100)
print("LOADED PHASE 3 OUTPUTS")
print("=" * 100)
print("Manifest rows:", len(manifest))
print("Manifest columns:", list(manifest.columns))
print("Splits rows:", len(splits))
print("Splits columns:", list(splits.columns))

required_manifest_cols = ["file", "subject_id", "fs_hz", "duration_s", "task_code", "trial_id", "fold_id"]
required_split_cols = [
    "file", "subject_id", "split", "type", "task_target", "label_action",
    "start_idx", "end_idx", "task_code", "trial_id", "fold_id"
]

missing_manifest = [c for c in required_manifest_cols if c not in manifest.columns]
missing_splits = [c for c in required_split_cols if c not in splits.columns]

if missing_manifest:
    raise ValueError(f"Manifest missing required columns: {missing_manifest}")

if missing_splits:
    raise ValueError(f"Splits missing required columns: {missing_splits}")

print("\n✅ Required manifest/splits columns are present.")

print("\nSubjects in manifest:")
print(sorted(manifest["subject_id"].astype(int).unique().tolist()))

print("\nSubjects in splits:")
print(sorted(splits["subject_id"].astype(int).unique().tolist()))

print("\nSplit distribution:")
print(splits["split"].value_counts(dropna=False))

print("\nWindow type distribution:")
print(splits["type"].value_counts(dropna=False))

print("\nTask-target distribution:")
print(splits["task_target"].value_counts(dropna=False).sort_index())

print("\nLabel-action distribution:")
print(splits["label_action"].value_counts(dropna=False).sort_index())

# Safety: Phase 3 should already have skipped missing Phase 1C files.
if len(manifest) == 1552:
    print("\n✅ Manifest contains 1552 valid files. The 104 missing Phase 1C files are excluded.")
else:
    print(f"\n⚠️ Manifest has {len(manifest)} files, not 1552. Continue only if this is expected.")


LOADED PHASE 3 OUTPUTS
Manifest rows: 1552
Manifest columns: ['file', 'subject_id', 'fs_hz', 'duration_s', 'task_code', 'trial_id', 'fold_id']
Splits rows: 475798
Splits columns: ['file', 'subject_id', 'split', 'type', 'task_target', 'label_action', 'start_idx', 'end_idx', 'task_code', 'trial_id', 'fold_id']

✅ Required manifest/splits columns are present.

Subjects in manifest:
[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]

Subjects in splits:
[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]

Split distribution:
split
train    411025
val       39731
test      25042
Name: count, dtype: int64

Window type distribution:
type
sliding    475798
Name: count, dtype: int64

Task-target distribution:
task_target
0     48222
1     90478
2    114627
3     78831
4     78603
5     65037
Name: count, dtype: int64

Label-action distribution:
label_action
0     48222
1    427576
Name: count, dtype: int64

✅ Manifest contains 1552 valid files. The 104 missin

In [14]:
# =====================================================================
# CELL 3 — Path resolution and cache helpers
# =====================================================================

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def safe_subject_sort_key(path_or_name):
    s = str(path_or_name)
    m = re.search(r"Sub-(\d+)", s)
    return int(m.group(1)) if m else 10**9


def resolve_label_csv(file_value, subject_id=None) -> Path | None:
    """
    Resolve Phase 2B labelonly CSV path from manifest/splits row.

    Does not create files. It only resolves existing paths.
    Handles:
      1) absolute path in 'file'
      2) relative path from ROOT_DIR
      3) filename-only with subject_id reconstruction
    """
    f = Path(str(file_value))

    # Absolute or current working dir path.
    if f.exists():
        return f.resolve()

    # Relative to ROOT_DIR.
    rel = ROOT_DIR / f
    if rel.exists():
        return rel.resolve()

    # Filename-only path; reconstruct from subject_id.
    if subject_id is not None and not pd.isna(subject_id):
        sid = int(subject_id)
        labelonly_dir = (
            ROOT_DIR
            / f"Sub-{sid}"
            / "cleaned"
            / "synchronized_proper_lite_union_v3"
            / "labelonly"
        )

        direct = labelonly_dir / f.name
        if direct.exists():
            return direct.resolve()

        matches = sorted(labelonly_dir.glob(f.name))
        if len(matches) == 1:
            return matches[0].resolve()

        # Last fallback inside subject folder.
        sub_dir = ROOT_DIR / f"Sub-{sid}"
        matches = sorted(sub_dir.glob(f"**/labelonly/{f.name}"))
        if len(matches) == 1:
            return matches[0].resolve()

    return None


def cache_path_from_label_csv(label_csv: Path) -> Path:
    return Path(str(label_csv) + NPZ_SUFFIX)


def load_npz(npz_path: Path) -> dict:
    with np.load(npz_path, allow_pickle=True) as z:
        out = {k: z[k] for k in z.files}

    # Convert channel arrays to Python lists.
    for k in [EEG_CH_KEY, EMG_CH_KEY, ET_CH_KEY, LABEL_CH_KEY]:
        if k in out:
            out[k] = [str(x) for x in out[k].tolist()]

    # Scalar metadata.
    for k in ["subject_id", "task", "trial"]:
        if k in out:
            try:
                out[k] = int(np.asarray(out[k]).item())
            except Exception:
                pass

    if "fs" in out:
        try:
            out["fs"] = float(np.asarray(out["fs"]).item())
        except Exception:
            pass

    return out


class NPZCacheLRU:
    def __init__(self, capacity=12):
        self.capacity = int(capacity)
        self._d: OrderedDict[str, dict] = OrderedDict()

    def get(self, file_value, subject_id=None):
        label_csv = resolve_label_csv(file_value, subject_id=subject_id)
        if label_csv is None:
            raise FileNotFoundError(f"Could not resolve label CSV from file={file_value}, subject_id={subject_id}")

        npz_path = cache_path_from_label_csv(label_csv)
        key = str(npz_path)

        if key in self._d:
            self._d.move_to_end(key)
            return self._d[key]

        if not npz_path.exists():
            raise FileNotFoundError(f"Phase 4 cache missing: {npz_path}")

        val = load_npz(npz_path)
        self._d[key] = val
        self._d.move_to_end(key)

        if len(self._d) > self.capacity:
            self._d.popitem(last=False)

        return val

    def clear(self):
        self._d.clear()


def validate_cache(npz: dict, npz_path: Path, expected_subject=None, expected_task=None, expected_trial=None):
    """
    Validate cache keys/shapes without creating or changing anything.
    """
    missing = [k for k in REQUIRED_CACHE_KEYS if k not in npz]
    if missing:
        return False, f"Missing cache keys: {missing}"

    n = len(npz["t"])

    checks = [
        (EEG_KEY, EEG_MASK_KEY, EEG_CH_KEY),
        (EMG_KEY, EMG_MASK_KEY, EMG_CH_KEY),
        (ET_KEY, ET_MASK_KEY, ET_CH_KEY),
    ]

    for x_key, m_key, ch_key in checks:
        X = npz[x_key]
        M = npz[m_key]
        ch = npz[ch_key]

        if X.ndim != 2:
            return False, f"{x_key} must be 2D, got shape {X.shape}"
        if M.ndim != 2:
            return False, f"{m_key} must be 2D, got shape {M.shape}"
        if X.shape != M.shape:
            return False, f"{x_key} shape {X.shape} != {m_key} shape {M.shape}"
        if X.shape[0] != n:
            return False, f"{x_key} rows {X.shape[0]} != t length {n}"
        if X.shape[1] != len(ch):
            return False, f"{x_key} channels {X.shape[1]} != len({ch_key}) {len(ch)}"

    labels = npz[LABEL_KEY]
    label_ch = npz[LABEL_CH_KEY]
    if labels.ndim != 2:
        return False, f"labels must be 2D, got shape {labels.shape}"
    if labels.shape[0] != n:
        return False, f"labels rows {labels.shape[0]} != t length {n}"
    if labels.shape[1] != len(label_ch):
        return False, f"labels channels {labels.shape[1]} != len(label_ch) {len(label_ch)}"

    if expected_subject is not None and int(npz["subject_id"]) != int(expected_subject):
        return False, f"Cache subject_id {npz['subject_id']} != expected {expected_subject}"

    if expected_task is not None and int(npz["task"]) != int(expected_task):
        return False, f"Cache task {npz['task']} != expected {expected_task}"

    if expected_trial is not None and int(npz["trial"]) != int(expected_trial):
        return False, f"Cache trial {npz['trial']} != expected {expected_trial}"

    if abs(float(npz["fs"]) - 250.0) > 1.0:
        return False, f"fs {npz['fs']} not close to 250 Hz"

    return True, "OK"

In [15]:
# =====================================================================
# CELL 4 — Input audit: files, caches, keys, shapes
# =====================================================================

def run_phase5_input_audit(manifest: pd.DataFrame, splits: pd.DataFrame):
    print("\n" + "=" * 100)
    print("PHASE 5 INPUT AUDIT")
    print("=" * 100)

    audit_rows = []
    manifest_file_set = set()

    for row in manifest.itertuples(index=False):
        label_csv = resolve_label_csv(row.file, subject_id=row.subject_id)
        manifest_file_set.add(str(row.file))

        if label_csv is None:
            audit_rows.append({
                "file": str(row.file),
                "subject_id": int(row.subject_id),
                "task_code": int(row.task_code),
                "trial_id": int(row.trial_id),
                "label_csv": "",
                "cache": "",
                "status": "LABEL_CSV_NOT_FOUND",
                "reason": "Could not resolve labelonly CSV",
            })
            continue

        npz_path = cache_path_from_label_csv(label_csv)

        if not npz_path.exists():
            audit_rows.append({
                "file": str(row.file),
                "subject_id": int(row.subject_id),
                "task_code": int(row.task_code),
                "trial_id": int(row.trial_id),
                "label_csv": str(label_csv),
                "cache": str(npz_path),
                "status": "CACHE_NOT_FOUND",
                "reason": "Phase 4 cache missing",
            })
            continue

        try:
            npz = load_npz(npz_path)
            ok, reason = validate_cache(
                npz,
                npz_path,
                expected_subject=int(row.subject_id),
                expected_task=int(row.task_code),
                expected_trial=int(row.trial_id),
            )

            status = "OK" if ok else "CACHE_INVALID"

            audit_rows.append({
                "file": str(row.file),
                "subject_id": int(row.subject_id),
                "task_code": int(row.task_code),
                "trial_id": int(row.trial_id),
                "label_csv": str(label_csv),
                "cache": str(npz_path),
                "status": status,
                "reason": reason,
                "n_samples": len(npz["t"]) if "t" in npz else np.nan,
                "fs": float(npz["fs"]) if "fs" in npz else np.nan,
                "eeg_shape": str(npz[EEG_KEY].shape) if EEG_KEY in npz else "",
                "emg_shape": str(npz[EMG_KEY].shape) if EMG_KEY in npz else "",
                "et_shape": str(npz[ET_KEY].shape) if ET_KEY in npz else "",
            })

        except Exception as e:
            audit_rows.append({
                "file": str(row.file),
                "subject_id": int(row.subject_id),
                "task_code": int(row.task_code),
                "trial_id": int(row.trial_id),
                "label_csv": str(label_csv),
                "cache": str(npz_path),
                "status": "CACHE_READ_ERROR",
                "reason": str(e),
            })

    audit_df = pd.DataFrame(audit_rows)

    print("\nAudit status counts:")
    print(audit_df["status"].value_counts(dropna=False))

    print("\nManifest file count:", len(manifest))
    print("Unique split file count:", splits["file"].nunique())

    # Check every split file appears resolvable too.
    split_unique = splits[["file", "subject_id"]].drop_duplicates()
    unresolved_splits = []
    for row in split_unique.itertuples(index=False):
        if resolve_label_csv(row.file, subject_id=row.subject_id) is None:
            unresolved_splits.append({"file": row.file, "subject_id": row.subject_id})

    print("Unresolved split file paths:", len(unresolved_splits))

    audit_path = AUDIT_DIR / "phase5_input_cache_audit.csv"
    audit_df.to_csv(audit_path, index=False)

    unresolved_path = AUDIT_DIR / "phase5_unresolved_split_files.csv"
    pd.DataFrame(unresolved_splits).to_csv(unresolved_path, index=False)

    print("\nSaved audit:", audit_path)
    print("Saved unresolved split files:", unresolved_path)

    bad = audit_df[audit_df["status"] != "OK"]

    if len(bad) == 0 and len(unresolved_splits) == 0:
        print("\n✅ PHASE 5 INPUT AUDIT PASSED")
        print("✅ All manifest labelonly files and Phase 4 caches are valid.")
        print("✅ Missing Phase 1C files are not present in manifest/splits.")
    else:
        print("\n❌ PHASE 5 INPUT AUDIT FAILED")
        display(bad.head(20))
        raise RuntimeError("Fix Phase 5 input audit failures before export.")

    return audit_df


if RUN_INPUT_AUDIT:
    phase5_input_audit_df = run_phase5_input_audit(manifest, splits)
else:
    print("RUN_INPUT_AUDIT=False, skipping input audit.")


PHASE 5 INPUT AUDIT

Audit status counts:
status
OK    1552
Name: count, dtype: int64

Manifest file count: 1552
Unique split file count: 1550
Unresolved split file paths: 0

Saved audit: /home/tsultan1/BioRob/Human Subject Data/_audit_phase5_exporter/phase5_input_cache_audit.csv
Saved unresolved split files: /home/tsultan1/BioRob/Human Subject Data/_audit_phase5_exporter/phase5_unresolved_split_files.csv

✅ PHASE 5 INPUT AUDIT PASSED
✅ All manifest labelonly files and Phase 4 caches are valid.
✅ Missing Phase 1C files are not present in manifest/splits.


In [16]:
# =====================================================================
# CELL 5 — Optional QC filter and split cleaning
# =====================================================================

def apply_optional_qc_filter(splits_df: pd.DataFrame) -> pd.DataFrame:
    """
    If qc_summary_v1.csv exists, keep only QC-accepted files.
    If it does not exist, return splits unchanged.
    """
    if not QC_CSV.exists():
        print("\n[QC] No qc_summary_v1.csv found. Proceeding without QC filtering.")
        return splits_df.copy()

    print("\n[QC] Loading:", QC_CSV)
    qc = pd.read_csv(QC_CSV)

    if "file" not in qc.columns:
        print("[QC] qc_summary_v1.csv has no 'file' column. Skipping QC filter.")
        return splits_df.copy()

    if "qc_keep" in qc.columns:
        qc_use = qc[qc["qc_keep"].astype(int) == 1].copy()
        print(f"[QC] Using qc_keep==1: {len(qc_use)}/{len(qc)} files kept.")
    elif "qc_flag" in qc.columns:
        bad_vals = {"bad", "reject", "fail", "drop"}
        qc_flag = qc["qc_flag"].astype(str).str.lower()
        qc_use = qc[~qc_flag.isin(bad_vals)].copy()
        print(f"[QC] Using qc_flag: {len(qc_use)}/{len(qc)} files kept.")
    else:
        print("[QC] No qc_keep/qc_flag column. Treating all QC rows as keep.")
        qc_use = qc.copy()

    # Match by basename so absolute/relative path differences do not break filtering.
    keep_bases = set(qc_use["file"].apply(lambda x: Path(str(x)).name).tolist())

    out = splits_df.copy()
    before = len(out)
    out["_file_base_tmp"] = out["file"].apply(lambda x: Path(str(x)).name)
    out = out[out["_file_base_tmp"].isin(keep_bases)].drop(columns=["_file_base_tmp"]).reset_index(drop=True)
    after = len(out)

    print(f"[QC] Windows before: {before}")
    print(f"[QC] Windows after : {after}")
    print(f"[QC] Windows dropped: {before - after}")

    return out


def clean_splits_for_export(splits_df: pd.DataFrame) -> pd.DataFrame:
    out = splits_df.copy()

    # Keep only requested window types.
    before = len(out)
    out = out[out["type"].isin(WINDOW_TYPES)].reset_index(drop=True)
    print(f"\nWindow type filter {WINDOW_TYPES}: {before} -> {len(out)} rows")

    # Ensure numeric columns have numeric types.
    numeric_cols = [
        "subject_id", "task_target", "label_action",
        "start_idx", "end_idx", "task_code", "trial_id", "fold_id"
    ]
    for c in numeric_cols:
        out[c] = pd.to_numeric(out[c], errors="raise").astype(int)

    # Valid split labels only.
    valid_splits = {"train", "val", "test"}
    bad_splits = sorted(set(out["split"].astype(str)) - valid_splits)
    if bad_splits:
        raise ValueError(f"Invalid split labels found: {bad_splits}")

    # Valid windows.
    bad_windows = out[out["end_idx"] <= out["start_idx"]]
    if len(bad_windows):
        raise ValueError(f"Found {len(bad_windows)} windows with end_idx <= start_idx.")

    # File resolution check has already been done, but create resolved file column internally.
    # This is not written to input files; it only helps cache lookup and export reproducibility.
    resolved = []
    for row in out.itertuples(index=False):
        p = resolve_label_csv(row.file, subject_id=row.subject_id)
        if p is None:
            raise FileNotFoundError(f"Could not resolve file for split row: {row.file}")
        resolved.append(str(p))

    out["resolved_file"] = resolved

    print("\nCleaned splits for export:")
    print("Rows:", len(out))
    print("Folds:", sorted(out["fold_id"].unique().tolist()))
    print("Splits:")
    print(out["split"].value_counts(dropna=False))
    print("Task targets:")
    print(out["task_target"].value_counts(dropna=False).sort_index())

    return out


splits_qc = apply_optional_qc_filter(splits)
splits_export = clean_splits_for_export(splits_qc)

# Save this exact export split table for reproducibility.
export_splits_path = AUDIT_DIR / "phase5_splits_used_for_export.csv"
splits_export.to_csv(export_splits_path, index=False)
print("\nSaved splits used for export:", export_splits_path)


[QC] No qc_summary_v1.csv found. Proceeding without QC filtering.

Window type filter ['sliding']: 475798 -> 475798 rows

Cleaned splits for export:
Rows: 475798
Folds: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Splits:
split
train    411025
val       39731
test      25042
Name: count, dtype: int64
Task targets:
task_target
0     48222
1     90478
2    114627
3     78831
4     78603
5     65037
Name: count, dtype: int64

Saved splits used for export: /home/tsultan1/BioRob/Human Subject Data/_audit_phase5_exporter/phase5_splits_used_for_export.csv


In [17]:
# =====================================================================
# CELL 6 — Stats/window helpers
# =====================================================================

def row_limit(split: str):
    if split == "train" and LIMIT_TRAIN:
        return LIMIT_TRAIN
    if split == "val" and LIMIT_VAL:
        return LIMIT_VAL
    if split == "test" and LIMIT_TEST:
        return LIMIT_TEST
    return None


def apply_window(arr: np.ndarray, s: int, e: int) -> np.ndarray:
    s = max(0, int(s))
    e = min(int(e), arr.shape[0])
    if e <= s:
        return arr[:0]
    return arr[s:e]


def new_stats(nc: int):
    return {
        "cnt": np.zeros(nc, dtype=np.float64),
        "sum": np.zeros(nc, dtype=np.float64),
        "sum2": np.zeros(nc, dtype=np.float64),
    }


def accumulate_stats(stats: dict, X: np.ndarray, M: np.ndarray):
    if X.size == 0:
        return

    X = np.nan_to_num(X.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    M = np.nan_to_num(M.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    M = (M > 0).astype(np.float64)

    cnt = M.sum(axis=0)
    if cnt.sum() <= 0:
        return

    stats["cnt"] += cnt
    stats["sum"] += (X * M).sum(axis=0)
    stats["sum2"] += ((X ** 2) * M).sum(axis=0)


def finalize_stats(stats: dict):
    cnt = np.maximum(stats["cnt"], 1.0)
    mean = stats["sum"] / cnt
    var = np.maximum(stats["sum2"] / cnt - mean ** 2, 0.0)
    std = np.sqrt(var) + 1e-8
    return mean.astype(np.float32), std.astype(np.float32)


def coverage_ok(M: np.ndarray, thr: float = MIN_COVERAGE) -> bool:
    if M is None or M.size == 0:
        return False
    M = np.nan_to_num(M.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    cov = (M > 0).mean(axis=0)
    return bool((cov >= thr).mean() > 0.5)


def normalize_window(X: np.ndarray, M: np.ndarray, mean: np.ndarray, std: np.ndarray):
    if X is None or X.size == 0:
        return None

    X = np.nan_to_num(X.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    M = np.nan_to_num(M.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    Y = (X - mean[None, :]) / std[None, :]
    Y[M <= 0] = 0.0
    Y = np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0)

    return Y.astype(np.float32)


def pad_or_trim_time(X: np.ndarray, target_len: int, n_channels: int) -> np.ndarray:
    """
    Return shape exactly (target_len, n_channels).
    If X is None, returns zeros.
    """
    if X is None or X.size == 0:
        return np.zeros((target_len, n_channels), dtype=np.float32)

    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 2:
        raise ValueError(f"Window must be 2D, got shape {X.shape}")

    T, C = X.shape
    if C != n_channels:
        raise ValueError(f"Window channels {C} != expected {n_channels}")

    if T == target_len:
        return X.astype(np.float32)
    if T > target_len:
        return X[:target_len].astype(np.float32)

    pad = np.zeros((target_len - T, C), dtype=np.float32)
    return np.concatenate([X, pad], axis=0).astype(np.float32)


def get_channel_info_from_npz(npz: dict):
    return {
        "EEG": list(npz.get(EEG_CH_KEY, [])),
        "EMG": list(npz.get(EMG_CH_KEY, [])),
        "ET": list(npz.get(ET_CH_KEY, [])),
        "label_ch": list(npz.get(LABEL_CH_KEY, [])),
    }

In [18]:
# =====================================================================
# CELL 7 — Balancing helpers
# =====================================================================

def analyze_task_distribution(df: pd.DataFrame, title="TASK DISTRIBUTION"):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print("Rows:", len(df))
    print("\nBy split:")
    print(df["split"].value_counts(dropna=False))
    print("\nlabel_action:")
    print(df["label_action"].value_counts(dropna=False).sort_index())
    print("\ntask_target:")
    print(df["task_target"].value_counts(dropna=False).sort_index())
    print("\nBy split and task_target:")
    print(pd.crosstab(df["split"], df["task_target"]))


def sample_group(g: pd.DataFrame, n: int, random_state: int):
    if len(g) == 0 or n <= 0:
        return g.iloc[:0].copy()
    replace = len(g) < n
    return g.sample(n=n, replace=replace, random_state=random_state)


def rebalance_train_rows(train_df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """
    Balanced supervised train export:
      - Equalizes non-zero task classes to target_per_action.
      - Caps task 0/REST so it does not dominate.
      - Only applied to TRAIN split.
      - Does not modify val/test.
    """
    if train_df.empty:
        return train_df.copy()

    task_counts = train_df["task_target"].value_counts().sort_index()
    nonzero_tasks = [int(t) for t in task_counts.index.tolist() if int(t) != 0]

    if len(nonzero_tasks) == 0:
        print("  [balance] No non-zero task targets found. Keeping train unchanged.")
        return train_df.copy()

    nonzero_counts = [int(task_counts.get(t, 0)) for t in nonzero_tasks]
    raw_target = min(nonzero_counts)
    max_target = ACTION_MAX_FACTOR * MIN_TASK_SAMPLES
    target_per_action = int(max(MIN_TASK_SAMPLES, min(raw_target, max_target)))

    balanced_parts = []

    print("  [balance] Original train task distribution:")
    for t, c in task_counts.items():
        print(f"    Task {int(t)}: {int(c)}")

    print(f"  [balance] Non-zero tasks: {nonzero_tasks}")
    print(f"  [balance] target_per_action: {target_per_action}")

    for t in nonzero_tasks:
        g = train_df[train_df["task_target"] == t]
        balanced_parts.append(sample_group(g, target_per_action, random_state + int(t)))

    action_total = target_per_action * len(nonzero_tasks)

    rest_df = train_df[train_df["task_target"] == 0]
    if len(rest_df) > 0:
        # Make rest at most roughly MAX_TASK0_RATIO of action total, but keep a minimum if possible.
        rest_target = int(min(len(rest_df), max(MIN_TASK_SAMPLES, round(MAX_TASK0_RATIO * action_total))))
        balanced_parts.append(sample_group(rest_df, rest_target, random_state + 999))
        print(f"  [balance] rest_target: {rest_target}")

    out = pd.concat(balanced_parts, ignore_index=True)
    out = out.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    print("  [balance] New train task distribution:")
    for t, c in out["task_target"].value_counts().sort_index().items():
        print(f"    Task {int(t)}: {int(c)}")

    return out


analyze_task_distribution(splits_export, title="SPLITS USED FOR PHASE 5 EXPORT")


SPLITS USED FOR PHASE 5 EXPORT
Rows: 475798

By split:
split
train    411025
val       39731
test      25042
Name: count, dtype: int64

label_action:
label_action
0     48222
1    427576
Name: count, dtype: int64

task_target:
task_target
0     48222
1     90478
2    114627
3     78831
4     78603
5     65037
Name: count, dtype: int64

By split and task_target:
task_target      0      1      2      3      4      5
split                                                
test          2538   4762   6033   4149   4137   3423
train        42914  77917  98383  68076  66789  56946
val           2770   7799  10211   6606   7677   4668


In [19]:
# =====================================================================
# CELL 8 — Export core
# =====================================================================

def fold_export_complete(fold_dir: Path) -> bool:
    """
    A fold is considered complete if stats_fold.json exists and split_meta.json
    exists for train/val/test where split folders exist.
    """
    if not fold_dir.exists():
        return False
    if not (fold_dir / "stats_fold.json").exists():
        return False

    for split in ["train", "val", "test"]:
        split_dir = fold_dir / split
        if not split_dir.exists():
            return False
        if not (split_dir / "split_meta.json").exists():
            return False
        shards = sorted(split_dir.glob(f"{split}_shard_*.npz"))
        if len(shards) == 0:
            return False

    return True


def prepare_fold_dir(fold_dir: Path):
    if fold_dir.exists() and OVERWRITE_EXPORTS:
        print(f"[overwrite] Removing old export folder: {fold_dir}")
        shutil.rmtree(fold_dir)
    ensure_dir(fold_dir)


def accumulate_train_stats(train_rows: pd.DataFrame, cache: NPZCacheLRU):
    """
    Compute train-only normalization stats from unbalanced train rows.
    """
    stats = {}
    channel_info = None

    def ensure_stat(key, nchan):
        if key not in stats:
            stats[key] = new_stats(nchan)

    print(f"  [pass1] Accumulating TRAIN stats over {len(train_rows):,} windows.")

    for r in train_rows.itertuples(index=False):
        npz = cache.get(r.resolved_file, subject_id=r.subject_id)

        if channel_info is None:
            channel_info = get_channel_info_from_npz(npz)

        s, e = int(r.start_idx), int(r.end_idx)

        if INCLUDE_EEG and EEG_KEY in npz and EEG_MASK_KEY in npz:
            X = apply_window(npz[EEG_KEY], s, e)
            M = apply_window(npz[EEG_MASK_KEY], s, e)
            if X.size > 0 and coverage_ok(M):
                ensure_stat(EEG_KEY, X.shape[1])
                accumulate_stats(stats[EEG_KEY], X, M)

        if INCLUDE_EMG and EMG_KEY in npz and EMG_MASK_KEY in npz:
            X = apply_window(npz[EMG_KEY], s, e)
            M = apply_window(npz[EMG_MASK_KEY], s, e)
            if X.size > 0 and coverage_ok(M):
                ensure_stat(EMG_KEY, X.shape[1])
                accumulate_stats(stats[EMG_KEY], X, M)

        if INCLUDE_ET and ET_KEY in npz and ET_MASK_KEY in npz:
            X = apply_window(npz[ET_KEY], s, e)
            M = apply_window(npz[ET_MASK_KEY], s, e)
            if X.size > 0 and coverage_ok(M):
                ensure_stat(ET_KEY, X.shape[1])
                accumulate_stats(stats[ET_KEY], X, M)

    means, stds = {}, {}
    for key in [EEG_KEY, EMG_KEY, ET_KEY]:
        if key in stats:
            means[key], stds[key] = finalize_stats(stats[key])

    if channel_info is None:
        raise RuntimeError("Could not determine channel info from train rows.")

    # Ensure all included modalities have stats.
    required_stats = []
    if INCLUDE_EEG:
        required_stats.append(EEG_KEY)
    if INCLUDE_EMG:
        required_stats.append(EMG_KEY)
    if INCLUDE_ET:
        required_stats.append(ET_KEY)

    missing_stats = [k for k in required_stats if k not in means]
    if missing_stats:
        raise RuntimeError(f"Could not compute train stats for modalities: {missing_stats}")

    return means, stds, channel_info


def export_split_rows(rows: pd.DataFrame, split: str, fold_dir: Path, cache: NPZCacheLRU,
                      means: dict, stds: dict, channel_info: dict, mode: str):
    out_split = fold_dir / split
    ensure_dir(out_split)

    n_eeg = len(channel_info["EEG"]) if INCLUDE_EEG else 0
    n_emg = len(channel_info["EMG"]) if INCLUDE_EMG else 0
    n_et = len(channel_info["ET"]) if INCLUDE_ET else 0

    shard_idx = 1
    buf = defaultdict(list)
    total_exported = 0
    total_skipped_no_modality = 0

    def flush():
        nonlocal shard_idx, buf

        if len(buf["y_action"]) == 0:
            return

        shard_path = out_split / f"{split}_shard_{shard_idx:04d}.npz"

        np.savez_compressed(
            shard_path,
            X_EEG=np.stack(buf["X_EEG"]).astype(np.float32),
            X_EMG=np.stack(buf["X_EMG"]).astype(np.float32),
            X_ET=np.stack(buf["X_ET"]).astype(np.float32),
            y_action=np.asarray(buf["y_action"], dtype=np.int8),
            y_task=np.asarray(buf["y_task"], dtype=np.int16),
            subject_id=np.asarray(buf["subject_id"], dtype=np.int16),
            task_code=np.asarray(buf["task_code"], dtype=np.int16),
            trial_id=np.asarray(buf["trial_id"], dtype=np.int16),
            fold_id=np.asarray(buf["fold_id"], dtype=np.int16),
            win_type=np.asarray(buf["win_type"], dtype=object),
            start_idx=np.asarray(buf["start_idx"], dtype=np.int32),
            end_idx=np.asarray(buf["end_idx"], dtype=np.int32),
            win_len=np.asarray(buf["win_len"], dtype=np.int32),
            source_file=np.asarray(buf["source_file"], dtype=object),
        )

        shard_idx += 1
        buf = defaultdict(list)

    print(f"  [pass2] Exporting {split}: {len(rows):,} windows.")
    print("  Distribution task_target:")
    for t, c in rows["task_target"].value_counts().sort_index().items():
        print(f"    Task {int(t)}: {int(c)}")

    for r in rows.itertuples(index=False):
        npz = cache.get(r.resolved_file, subject_id=r.subject_id)
        s, e = int(r.start_idx), int(r.end_idx)
        win_len = max(0, e - s)

        Xe = Xm = Xt = None

        if INCLUDE_EEG:
            X = apply_window(npz[EEG_KEY], s, e)
            M = apply_window(npz[EEG_MASK_KEY], s, e)
            if X.size > 0 and coverage_ok(M) and EEG_KEY in means:
                Xe = normalize_window(X, M, means[EEG_KEY], stds[EEG_KEY])

        if INCLUDE_EMG:
            X = apply_window(npz[EMG_KEY], s, e)
            M = apply_window(npz[EMG_MASK_KEY], s, e)
            if X.size > 0 and coverage_ok(M) and EMG_KEY in means:
                Xm = normalize_window(X, M, means[EMG_KEY], stds[EMG_KEY])

        if INCLUDE_ET:
            X = apply_window(npz[ET_KEY], s, e)
            M = apply_window(npz[ET_MASK_KEY], s, e)
            if X.size > 0 and coverage_ok(M) and ET_KEY in means:
                Xt = normalize_window(X, M, means[ET_KEY], stds[ET_KEY])

        # If all modalities fail coverage, skip this window.
        if (Xe is None) and (Xm is None) and (Xt is None):
            total_skipped_no_modality += 1
            continue

        Xe = pad_or_trim_time(Xe, FIXED_WINDOW_LEN, n_eeg)
        Xm = pad_or_trim_time(Xm, FIXED_WINDOW_LEN, n_emg)
        Xt = pad_or_trim_time(Xt, FIXED_WINDOW_LEN, n_et)

        # Final hard safety.
        Xe = np.nan_to_num(Xe, nan=0.0, posinf=0.0, neginf=0.0)
        Xm = np.nan_to_num(Xm, nan=0.0, posinf=0.0, neginf=0.0)
        Xt = np.nan_to_num(Xt, nan=0.0, posinf=0.0, neginf=0.0)

        buf["X_EEG"].append(Xe)
        buf["X_EMG"].append(Xm)
        buf["X_ET"].append(Xt)
        buf["y_action"].append(int(r.label_action))
        buf["y_task"].append(int(r.task_target))
        buf["subject_id"].append(int(r.subject_id))
        buf["task_code"].append(int(r.task_code))
        buf["trial_id"].append(int(r.trial_id))
        buf["fold_id"].append(int(r.fold_id))
        buf["win_type"].append(str(r.type))
        buf["start_idx"].append(int(r.start_idx))
        buf["end_idx"].append(int(r.end_idx))
        buf["win_len"].append(int(win_len))
        buf["source_file"].append(str(r.resolved_file))

        total_exported += 1

        if len(buf["y_action"]) >= SHARD_SIZE:
            flush()

    flush()

    meta = {
        "split": split,
        "mode": mode,
        "num_input_windows": int(len(rows)),
        "num_exported_windows": int(total_exported),
        "num_skipped_no_modality": int(total_skipped_no_modality),
        "shard_size": int(SHARD_SIZE),
        "fixed_window_len": int(FIXED_WINDOW_LEN),
        "min_coverage": float(MIN_COVERAGE),
        "task_distribution_input": {str(k): int(v) for k, v in rows["task_target"].value_counts().sort_index().items()},
        "label_action_distribution_input": {str(k): int(v) for k, v in rows["label_action"].value_counts().sort_index().items()},
    }

    (out_split / "split_meta.json").write_text(json.dumps(meta, indent=2))

    return {
        "split": split,
        "mode": mode,
        "input_windows": int(len(rows)),
        "exported_windows": int(total_exported),
        "skipped_no_modality": int(total_skipped_no_modality),
        "shards": int(len(list(out_split.glob(f'{split}_shard_*.npz')))),
    }


def export_fold(splits_df: pd.DataFrame, fold_id: int, mode: str):
    if mode not in ["balanced", "ssl"]:
        raise ValueError("mode must be 'balanced' or 'ssl'")

    out_name = OUT_NAME_BALANCED if mode == "balanced" else OUT_NAME_SSL
    fold_dir = DATASET_DIR / f"{out_name}_fold{fold_id}"

    print("\n" + "=" * 100)
    print(f"EXPORT FOLD {fold_id} | mode={mode}")
    print("=" * 100)
    print("Output:", fold_dir)

    if fold_export_complete(fold_dir) and not OVERWRITE_EXPORTS:
        print("✅ Existing complete fold export found, skipping.")
        return {
            "fold_id": int(fold_id),
            "mode": mode,
            "status": "SKIP_EXISTING_COMPLETE",
            "fold_dir": str(fold_dir),
        }

    prepare_fold_dir(fold_dir)

    fold_rows = splits_df[splits_df["fold_id"] == int(fold_id)].copy()
    if len(fold_rows) == 0:
        print("No rows for this fold.")
        return {
            "fold_id": int(fold_id),
            "mode": mode,
            "status": "NO_ROWS",
            "fold_dir": str(fold_dir),
        }

    train_stats_rows = fold_rows[fold_rows["split"] == "train"].copy()
    if len(train_stats_rows) == 0:
        raise RuntimeError(f"Fold {fold_id} has no train rows.")

    lim = row_limit("train")
    if lim:
        train_stats_rows = train_stats_rows.iloc[:lim].copy()

    print("\nFold rows by split:")
    print(fold_rows["split"].value_counts(dropna=False))

    print("\nTrain stats task distribution before balancing:")
    print(train_stats_rows["task_target"].value_counts().sort_index())

    cache = NPZCacheLRU(capacity=12)

    means, stds, channel_info = accumulate_train_stats(train_stats_rows, cache)

    # Save fold stats.
    stats_meta = {
        "fold_id": int(fold_id),
        "mode": mode,
        "root_dir": str(ROOT_DIR),
        "dataset_dir": str(DATASET_DIR),
        "splits_csv": str(SPLITS_CSV),
        "manifest_csv": str(MANIFEST_CSV),
        "window_types": WINDOW_TYPES,
        "fixed_window_len": int(FIXED_WINDOW_LEN),
        "min_coverage": float(MIN_COVERAGE),
        "include": {"EEG": INCLUDE_EEG, "EMG": INCLUDE_EMG, "ET": INCLUDE_ET},
        "channels": channel_info,
        "stats": {},
        "balancing": {
            "applied_to_train": bool(mode == "balanced"),
            "max_task0_ratio": float(MAX_TASK0_RATIO),
            "min_task_samples": int(MIN_TASK_SAMPLES),
            "action_max_factor": int(ACTION_MAX_FACTOR),
        },
    }

    for key in [EEG_KEY, EMG_KEY, ET_KEY]:
        stats_meta["stats"][key] = {
            "mean": means[key].tolist(),
            "std": stds[key].tolist(),
        }

    (fold_dir / "stats_fold.json").write_text(json.dumps(stats_meta, indent=2))

    split_summaries = []

    for split in ["train", "val", "test"]:
        rows = fold_rows[fold_rows["split"] == split].copy()

        lim = row_limit(split)
        if lim:
            rows = rows.iloc[:lim].copy()

        if split == "train" and mode == "balanced":
            print("\nApplying TRAIN-only balancing for supervised export.")
            rows = rebalance_train_rows(rows, random_state=42 + int(fold_id))
        else:
            print(f"\nKeeping original {split} distribution for mode={mode}.")

        if len(rows) == 0:
            print(f"  No rows for split={split}.")
            continue

        summary = export_split_rows(rows, split, fold_dir, cache, means, stds, channel_info, mode=mode)
        split_summaries.append(summary)

    cache.clear()

    summary_path = fold_dir / "export_summary.json"
    summary_json = {
        "fold_id": int(fold_id),
        "mode": mode,
        "fold_dir": str(fold_dir),
        "split_summaries": split_summaries,
    }
    summary_path.write_text(json.dumps(summary_json, indent=2))

    print(f"\n✅ Fold {fold_id} export finished | mode={mode}")
    print("Summary:", summary_path)

    return {
        "fold_id": int(fold_id),
        "mode": mode,
        "status": "OK",
        "fold_dir": str(fold_dir),
        "summary_json": str(summary_path),
    }

In [20]:
# =====================================================================
# CELL 8.5 — Clean old Phase 5 exports before final overwrite run
# ---------------------------------------------------------------------
# This function was missing in the previous notebook version.
# It safely backs up/removes old exports_v1_balanced_foldK and
# exports_v1_ssl_foldK folders before rebuilding sliding-only exports.
# =====================================================================

from datetime import datetime
import shutil

def clean_old_phase5_exports_for_final_run(folds_to_run):
    """
    Clean old Phase 5 export folders before rebuilding.

    Why this is needed:
    - np.savez can overwrite same filenames, but old shard files can remain
      if the new run produces fewer shards.
    - After changing to WINDOW_TYPES=['sliding'], we should not mix old
      sliding+onset_anchor shards with new sliding-only shards.
    """
    if not OVERWRITE_EXPORTS:
        print("\n[CLEAN] OVERWRITE_EXPORTS=False -> keeping existing export folders.")
        return

    if not CLEAN_PHASE5_EXPORTS_BEFORE_RUN:
        print("\n[CLEAN] CLEAN_PHASE5_EXPORTS_BEFORE_RUN=False -> no pre-cleaning.")
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_root = DATASET_DIR / f"_backup_phase5_exports_{timestamp}"

    candidates = []
    for fold_id in folds_to_run:
        fold_id = int(fold_id)

        if EXPORT_BALANCED:
            candidates.append(DATASET_DIR / f"{OUT_NAME_BALANCED}_fold{fold_id}")

        if EXPORT_SSL:
            candidates.append(DATASET_DIR / f"{OUT_NAME_SSL}_fold{fold_id}")

    existing = [p for p in candidates if p.exists()]

    print("\n" + "=" * 100)
    print("PHASE 5 PRE-CLEAN / OVERWRITE CONTROL")
    print("=" * 100)
    print("Folds to clean:", list(map(int, folds_to_run)))
    print("Existing export folders found:", len(existing))
    print("Backup old exports:", BACKUP_OLD_EXPORTS_BEFORE_OVERWRITE)

    if len(existing) == 0:
        print("✅ No old Phase 5 export folders found for these folds.")
        return

    if BACKUP_OLD_EXPORTS_BEFORE_OVERWRITE:
        backup_root.mkdir(parents=True, exist_ok=True)
        print("Backup root:", backup_root)

    for src in existing:
        if BACKUP_OLD_EXPORTS_BEFORE_OVERWRITE:
            dst = backup_root / src.name

            # Avoid accidental overwrite if notebook is rerun in the same second.
            if dst.exists():
                k = 1
                while (backup_root / f"{src.name}_{k}").exists():
                    k += 1
                dst = backup_root / f"{src.name}_{k}"

            print("Backing up:", src, "->", dst)
            shutil.move(str(src), str(dst))
        else:
            print("Deleting:", src)
            shutil.rmtree(src)

    print("✅ Old Phase 5 export folders removed from active dataset folder.")


In [21]:
# =====================================================================
# CELL 9 — Run Phase 5 export
# =====================================================================

def run_phase5_exports():
    folds = sorted([int(x) for x in splits_export["fold_id"].unique().tolist()])

    if RUN_ONE_FOLD_FIRST:
        folds_to_run = [int(ONE_FOLD_ID)]
    elif RUN_ALL_FOLDS:
        folds_to_run = folds
    else:
        print("No export selected. Set RUN_ONE_FOLD_FIRST=True or RUN_ALL_FOLDS=True.")
        return pd.DataFrame()

    print("\n" + "=" * 100)
    print("RUNNING PHASE 5 EXPORT")
    print("=" * 100)
    print("Folds available:", folds)
    print("Folds to run   :", folds_to_run)
    print("EXPORT_BALANCED:", EXPORT_BALANCED)
    print("EXPORT_SSL     :", EXPORT_SSL)
    print("OVERWRITE      :", OVERWRITE_EXPORTS)
    print("WINDOW_TYPES   :", WINDOW_TYPES)

    clean_old_phase5_exports_for_final_run(folds_to_run)

    results = []

    for fold_id in folds_to_run:
        if EXPORT_BALANCED:
            results.append(export_fold(splits_export, fold_id=fold_id, mode="balanced"))
        if EXPORT_SSL:
            results.append(export_fold(splits_export, fold_id=fold_id, mode="ssl"))

    result_df = pd.DataFrame(results)
    out_path = AUDIT_DIR / "phase5_export_run_summary.csv"
    result_df.to_csv(out_path, index=False)

    print("\n" + "=" * 100)
    print("PHASE 5 EXPORT RUN SUMMARY")
    print("=" * 100)
    if len(result_df):
        print(result_df["status"].value_counts(dropna=False))
    print("Saved:", out_path)

    return result_df


phase5_export_summary = run_phase5_exports()


RUNNING PHASE 5 EXPORT
Folds available: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Folds to run   : [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
EXPORT_BALANCED: True
EXPORT_SSL     : True
OVERWRITE      : True
WINDOW_TYPES   : ['sliding']

PHASE 5 PRE-CLEAN / OVERWRITE CONTROL
Folds to clean: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Existing export folders found: 38
Backup old exports: True
Backup root: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/_backup_phase5_exports_20260526_005256
Backing up: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/exports_v1_balanced_fold1 -> /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/_backup_phase5_exports_20260526_005256/exports_v1_balanced_fold1
Backing up: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/exports_v1_ssl_fold1 -> /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/_backup_phase5_exports_20260526_005256/exp

In [22]:
# =====================================================================
# CELL 10 — Verify exported shards and inspect one shard
# =====================================================================

def verify_export_folder(fold_dir: Path):
    rows = []

    for split in ["train", "val", "test"]:
        split_dir = fold_dir / split
        meta_path = split_dir / "split_meta.json"

        shards = sorted(split_dir.glob(f"{split}_shard_*.npz")) if split_dir.exists() else []

        num_windows = 0
        shapes = []

        for shard in shards:
            try:
                with np.load(shard, allow_pickle=True) as z:
                    n = len(z["y_action"])
                    num_windows += n
                    shapes.append({
                        "shard": shard.name,
                        "N": int(n),
                        "X_EEG": str(z["X_EEG"].shape),
                        "X_EMG": str(z["X_EMG"].shape),
                        "X_ET": str(z["X_ET"].shape),
                        "y_action": str(z["y_action"].shape),
                        "y_task": str(z["y_task"].shape),
                        "nan_eeg": int(np.isnan(z["X_EEG"]).sum()),
                        "nan_emg": int(np.isnan(z["X_EMG"]).sum()),
                        "nan_et": int(np.isnan(z["X_ET"]).sum()),
                    })
            except Exception as e:
                shapes.append({"shard": shard.name, "error": str(e)})

        rows.append({
            "fold_dir": str(fold_dir),
            "split": split,
            "split_dir_exists": split_dir.exists(),
            "split_meta_exists": meta_path.exists(),
            "num_shards": len(shards),
            "num_windows": int(num_windows),
            "first_shard_shape": json.dumps(shapes[0]) if shapes else "",
        })

    return pd.DataFrame(rows)


def verify_phase5_outputs():
    rows = []

    for fold_id in sorted([int(x) for x in splits_export["fold_id"].unique().tolist()]):
        for out_name in [OUT_NAME_BALANCED, OUT_NAME_SSL]:
            fold_dir = DATASET_DIR / f"{out_name}_fold{fold_id}"
            if fold_dir.exists():
                dfv = verify_export_folder(fold_dir)
                dfv["fold_id"] = int(fold_id)
                dfv["export_name"] = out_name
                rows.append(dfv)

    if not rows:
        print("No Phase 5 export folders found yet.")
        return pd.DataFrame()

    check_df = pd.concat(rows, ignore_index=True)
    out_path = AUDIT_DIR / "phase5_export_verification.csv"
    check_df.to_csv(out_path, index=False)

    print("\n" + "=" * 100)
    print("PHASE 5 EXPORT VERIFICATION")
    print("=" * 100)
    print("Saved:", out_path)
    display(check_df.head(20))

    return check_df


phase5_export_check = verify_phase5_outputs()


# Inspect one shard for the latest run.
def inspect_one_phase5_shard():
    candidates = []

    for out_name in [OUT_NAME_BALANCED, OUT_NAME_SSL]:
        for fold_id in sorted([int(x) for x in splits_export["fold_id"].unique().tolist()]):
            fold_dir = DATASET_DIR / f"{out_name}_fold{fold_id}"
            shard = fold_dir / "train" / "train_shard_0001.npz"
            if shard.exists():
                candidates.append(shard)

    if not candidates:
        print("No train shard found to inspect.")
        return

    shard = candidates[0]
    print("\n" + "=" * 100)
    print("INSPECTING ONE PHASE 5 SHARD")
    print("=" * 100)
    print("Shard:", shard)

    with np.load(shard, allow_pickle=True) as z:
        print("Keys:", z.files)
        print("X_EEG:", z["X_EEG"].shape, z["X_EEG"].dtype)
        print("X_EMG:", z["X_EMG"].shape, z["X_EMG"].dtype)
        print("X_ET :", z["X_ET"].shape, z["X_ET"].dtype)
        print("y_action:", z["y_action"].shape, "values:", np.unique(z["y_action"], return_counts=True))
        print("y_task  :", z["y_task"].shape, "values:", np.unique(z["y_task"], return_counts=True))
        print("subject_id:", z["subject_id"].shape, "unique sample:", np.unique(z["subject_id"])[:10])
        print("Any NaN EEG:", bool(np.isnan(z["X_EEG"]).any()))
        print("Any NaN EMG:", bool(np.isnan(z["X_EMG"]).any()))
        print("Any NaN ET :", bool(np.isnan(z["X_ET"]).any()))

    print("\n✅ One shard inspection completed.")


inspect_one_phase5_shard()


PHASE 5 EXPORT VERIFICATION
Saved: /home/tsultan1/BioRob/Human Subject Data/_audit_phase5_exporter/phase5_export_verification.csv


,fold_dir,split,split_dir_exists,split_meta_exists,num_shards,num_windows,first_shard_shape,fold_id,export_name
0,/home/tsultan1/BioRob/Human Subject Data/_data...,train,True,True,1,900,"{""shard"": ""train_shard_0001.npz"", ""N"": 900, ""X...",1,exports_v1_balanced
1,/home/tsultan1/BioRob/Human Subject Data/_data...,val,True,True,1,2507,"{""shard"": ""val_shard_0001.npz"", ""N"": 2507, ""X_...",1,exports_v1_balanced
2,/home/tsultan1/BioRob/Human Subject Data/_data...,test,True,True,1,2068,"{""shard"": ""test_shard_0001.npz"", ""N"": 2068, ""X...",1,exports_v1_balanced
3,/home/tsultan1/BioRob/Human Subject Data/_data...,train,True,True,5,20467,"{""shard"": ""train_shard_0001.npz"", ""N"": 5000, ""...",1,exports_v1_ssl
4,/home/tsultan1/BioRob/Human Subject Data/_data...,val,True,True,1,2507,"{""shard"": ""val_shard_0001.npz"", ""N"": 2507, ""X_...",1,exports_v1_ssl
5,/home/tsultan1/BioRob/Human Subject Data/_data...,test,True,True,1,2068,"{""shard"": ""test_shard_0001.npz"", ""N"": 2068, ""X...",1,exports_v1_ssl
6,/home/tsultan1/BioRob/Human Subject Data/_data...,train,True,True,1,900,"{""shard"": ""train_shard_0001.npz"", ""N"": 900, ""X...",2,exports_v1_balanced
7,/home/tsultan1/BioRob/Human Subject Data/_data...,val,True,True,1,2068,"{""shard"": ""val_shard_0001.npz"", ""N"": 2068, ""X_...",2,exports_v1_balanced
8,/home/tsultan1/BioRob/Human Subject Data/_data...,test,True,True,1,2507,"{""shard"": ""test_shard_0001.npz"", ""N"": 2507, ""X...",2,exports_v1_balanced
9,/home/tsultan1/BioRob/Human Subject Data/_data...,train,True,True,5,20467,"{""shard"": ""train_shard_0001.npz"", ""N"": 5000, ""...",2,exports_v1_ssl



INSPECTING ONE PHASE 5 SHARD
Shard: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/exports_v1_balanced_fold1/train/train_shard_0001.npz
Keys: ['X_EEG', 'X_EMG', 'X_ET', 'y_action', 'y_task', 'subject_id', 'task_code', 'trial_id', 'fold_id', 'win_type', 'start_idx', 'end_idx', 'win_len', 'source_file']
X_EEG: (900, 500, 8) float32
X_EMG: (900, 500, 4) float32
X_ET : (900, 500, 17) float32
y_action: (900,) values: (array([0, 1], dtype=int8), array([150, 750]))
y_task  : (900,) values: (array([0, 1, 2, 3, 4, 5], dtype=int16), array([150, 150, 150, 150, 150, 150]))
subject_id: (900,) unique sample: [ 3  5  6  7  8  9 10 11 12 14]
Any NaN EEG: False
Any NaN EMG: False
Any NaN ET : False

✅ One shard inspection completed.


In [23]:
# =====================================================================
# CELL 11 — Final instructions
# =====================================================================

print("\n" + "=" * 100)
print("PHASE 5 NOTE")
print("=" * 100)
print("This final version is configured for all-fold sliding-only overwrite rerun. If you want one-fold testing, set:")
print("  RUN_ONE_FOLD_FIRST = False")
print("  RUN_ALL_FOLDS = True")
print("Then rerun the export cell.")
print()
print("The 104 missing Phase 1C files are not used because Phase 5 only reads Phase 3 splits/manifest.")
print("Phase 5 input files:", len(manifest))
print("Phase 5 split windows:", len(splits_export))


PHASE 5 NOTE
This final version is configured for all-fold sliding-only overwrite rerun. If you want one-fold testing, set:
  RUN_ONE_FOLD_FIRST = False
  RUN_ALL_FOLDS = True
Then rerun the export cell.

The 104 missing Phase 1C files are not used because Phase 5 only reads Phase 3 splits/manifest.
Phase 5 input files: 1552
Phase 5 split windows: 475798
